In [36]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [39]:
batch_size = 16
IMG_SIZE = (224, 224)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.


In [4]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


In [5]:
for layer in base_model.layers[-80:]:
    layer.trainable = True



In [6]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]


In [7]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
Dense(512, activation="relu"),
Dropout(0.6),
Dense(256, activation="relu"),
Dropout(0.5)

x = Dropout(0.5)(x)
output = Dense(3, activation="softmax")(x)  # covid, normal, pneumonia

model = Model(inputs=base_model.input, outputs=output)


In [8]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [10]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)


Epoch 1/15


ValueError: Creating variables on a non-first call to a function decorated with tf.function.

In [41]:
from tensorflow.keras.losses import CategoricalCrossentropy

model.compile(
    optimizer=Adam(1e-5),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)



In [21]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(1.0091838864537674), 1: np.float64(0.9954704550133827), 2: np.float64(0.9954704550133827)}


In [22]:
model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks
)



Epoch 1/30
 24/303 ━━━━━━━━━━━━━━━━━━━━ 11:50 3s/step - accuracy: 0.3284 - loss: 1.8274

KeyboardInterrupt: 

In [23]:
test_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test",
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
model.evaluate(test_gen)


Found 1036 images belonging to 3 classes.
33/65 ━━━━━━━━━━━━━━━━━━━━ 13s 424ms/step - accuracy: 0.1078 - loss: 2.0268

KeyboardInterrupt: 

In [40]:
import tensorflow as tf
tf.keras.backend.clear_session()


In [20]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


In [42]:
for layer in base_model.layers:
    layer.trainable = False


In [45]:
from tensorflow.keras.layers import BatchNormalization

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

output = Dense(3, activation="softmax")(x)
model = Model(base_model.input, output)



In [46]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [47]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        "best_model.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]


In [48]:
model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks   # ✅ AGAIN HERE
)


Epoch 1/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 499ms/step - accuracy: 0.5982 - loss: 1.0767
Epoch 1: val_accuracy improved from None to 0.80989, saving model to best_model.h5



Epoch 1: finished saving model to best_model.h5
303/303 ━━━━━━━━━━━━━━━━━━━━ 190s 613ms/step - accuracy: 0.6904 - loss: 0.8687 - val_accuracy: 0.8099 - val_loss: 0.5064 - learning_rate: 1.0000e-04
Epoch 2/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 627ms/step - accuracy: 0.7877 - loss: 0.6486
Epoch 2: val_accuracy improved from 0.80989 to 0.81571, saving model to best_model.h5



Epoch 2: finished saving model to best_model.h5
303/303 ━━━━━━━━━━━━━━━━━━━━ 225s 744ms/step - accuracy: 0.7909 - loss: 0.6300 - val_accuracy: 0.8157 - val_loss: 0.5233 - learning_rate: 1.0000e-04
Epoch 3/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 607ms/step - accuracy: 0.8141 - loss: 0.5400
Epoch 3: val_accuracy did not improve from 0.81571
303/303 ━━━━━━━━━━━━━━━━━━━━ 220s 726ms/step - accuracy: 0.8194 - loss: 0.5220 - val_accuracy: 0.7730 - val_loss: 0.5855 - learning_rate: 1.0000e-04
Epoch 4/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 622ms/step - accuracy: 0.8296 - loss: 0.5216
Epoch 4: val_accuracy improved from 0.81571 to 0.82444, saving model to best_model.h5



Epoch 4: finished saving model to best_model.h5
303/303 ━━━━━━━━━━━━━━━━━━━━ 225s 743ms/step - accuracy: 0.8273 - loss: 0.5234 - val_accuracy: 0.8244 - val_loss: 0.4891 - learning_rate: 1.0000e-04
Epoch 5/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 597ms/step - accuracy: 0.8326 - loss: 0.4906
Epoch 5: val_accuracy improved from 0.82444 to 0.83608, saving model to best_model.h5



Epoch 5: finished saving model to best_model.h5
303/303 ━━━━━━━━━━━━━━━━━━━━ 220s 727ms/step - accuracy: 0.8385 - loss: 0.4776 - val_accuracy: 0.8361 - val_loss: 0.4678 - learning_rate: 1.0000e-04
Epoch 6/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 632ms/step - accuracy: 0.8472 - loss: 0.4327
Epoch 6: val_accuracy did not improve from 0.83608
303/303 ━━━━━━━━━━━━━━━━━━━━ 230s 760ms/step - accuracy: 0.8395 - loss: 0.4617 - val_accuracy: 0.8215 - val_loss: 0.4961 - learning_rate: 1.0000e-04
Epoch 7/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 622ms/step - accuracy: 0.8489 - loss: 0.4327
Epoch 7: val_accuracy did not improve from 0.83608
303/303 ━━━━━━━━━━━━━━━━━━━━ 225s 742ms/step - accuracy: 0.8469 - loss: 0.4280 - val_accuracy: 0.7953 - val_loss: 0.5509 - learning_rate: 1.0000e-04
Epoch 8/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 616ms/step - accuracy: 0.8511 - loss: 0.4496
Epoch 8: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.

Epoch 8: val_accuracy did not improve from 0.83608
303/303 ━

In [49]:
for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-40:]:  # last dense block
    layer.trainable = True


In [50]:
loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)


In [51]:
model.compile(
    optimizer=Adam(1e-5),
    loss=loss,
    metrics=["accuracy"]
)



In [52]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(1.0091838864537674), 1: np.float64(0.9954704550133827), 2: np.float64(0.9954704550133827)}


In [53]:
model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks,
    class_weight=class_weights
)


Epoch 1/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 620ms/step - accuracy: 0.8220 - loss: 0.7810
Epoch 1: val_accuracy did not improve from 0.83608
303/303 ━━━━━━━━━━━━━━━━━━━━ 234s 753ms/step - accuracy: 0.8281 - loss: 0.7673 - val_accuracy: 0.7507 - val_loss: 0.9685 - learning_rate: 1.0000e-05
Epoch 2/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 642ms/step - accuracy: 0.8288 - loss: 0.7686
Epoch 2: val_accuracy did not improve from 0.83608
303/303 ━━━━━━━━━━━━━━━━━━━━ 230s 760ms/step - accuracy: 0.8240 - loss: 0.7682 - val_accuracy: 0.7391 - val_loss: 0.9830 - learning_rate: 1.0000e-05
Epoch 3/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 651ms/step - accuracy: 0.8229 - loss: 0.7574
Epoch 3: ReduceLROnPlateau reducing learning rate to 2.9999999242136253e-06.

Epoch 3: val_accuracy did not improve from 0.83608
303/303 ━━━━━━━━━━━━━━━━━━━━ 236s 778ms/step - accuracy: 0.8186 - loss: 0.7652 - val_accuracy: 0.7265 - val_loss: 0.9725 - learning_rate: 1.0000e-05
Epoch 4/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 0s 637ms/step - ac

In [54]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=test_gen.class_indices.keys()))


NameError: name 'test_gen' is not defined

In [55]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False   # IMPORTANT for correct labels
)


Found 1036 images belonging to 3 classes.


In [56]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print(classification_report(
    y_true,
    y_pred,
    target_names=list(test_gen.class_indices.keys())
))


65/65 ━━━━━━━━━━━━━━━━━━━━ 34s 491ms/step
              precision    recall  f1-score   support

       covid       0.94      0.94      0.94       340
      normal       0.86      0.93      0.89       348
   pneumonia       0.96      0.88      0.92       348

    accuracy                           0.92      1036
   macro avg       0.92      0.92      0.92      1036
weighted avg       0.92      0.92      0.92      1036



In [57]:
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True


In [58]:
model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)


In [ ]:
model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    callbacks=callbacks,
    class_weight=class_weights
)


Epoch 1/30
  4/303 ━━━━━━━━━━━━━━━━━━━━ 2:27 494ms/step - accuracy: 0.8802 - loss: 0.7046